In [1]:
import os
import sys
import time

In [2]:
import importlib
from pathlib import Path

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import library.uart3_protocol as uart3_protocol
importlib.reload(uart3_protocol)
from library.uart3_protocol import UART3Protocol

print("Library updated / reloaded")

Library updated / reloaded


In [3]:
PORT = "/dev/cu.usbmodem1103"
BAUD = 1300000 # 1900000 有機會，但很常有 error
MAX_BUFFER_SIZE = 32768
TIMEOUT = 0.01

uart = UART3Protocol(
    port = PORT,
    baud = BAUD,
    max_buffer_size = MAX_BUFFER_SIZE,
    timeout = TIMEOUT
)

In [4]:
try:
    uart.open()
    print("open success")

except Exception as e:
    print("open fail")
    print(e)

open success


In [5]:
uart.write_line("test 123123")
uart.read_line()

'test 123123'

In [6]:
TEST_SIZES = [
    16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768,
]

In [7]:
print(f"PORT = {PORT}")
print(f"BAUD = {BAUD}")
print(
    f"{'Size(bytes)':>12} | "
    f"{'Echo Len':>8} | "
    f"{'OK':>5} | "
    f"{'Time(s)':>10} | "
    f"{'Speed(KB/s)':>12}"
)
print("-" * 62)

for size in TEST_SIZES:
    msg = os.urandom(size)

    try:
        uart.write_line("ECHO_PACKET")

        ready = uart.read_line(timeout=2.0)
        if ready != "READY":
            raise RuntimeError(f"Expected READY, got: {ready!r}")

        start = time.perf_counter()

        uart.send_packet(msg)
        echo = uart.receive_packet()

        end = time.perf_counter()

        elapsed = end - start
        ok = echo == msg

        total_bytes = 2 + size + 1 + 2 + size
        speed_kbs = total_bytes / elapsed / 1024

        print(
            f"{size:12d} | "
            f"{len(echo):8d} | "
            f"{str(ok):>5} | "
            f"{elapsed:10.6f} | "
            f"{speed_kbs:12.2f}"
        )

    except Exception as e:
        print(
            f"{size:12d} | "
            f"{'ERROR':>8} | "
            f"{'False':>5} | "
            f"{'-':>10} | "
            f"{str(e)}"
        )

    time.sleep(0.05)

PORT = /dev/cu.usbmodem1103
BAUD = 1300000
 Size(bytes) | Echo Len |    OK |    Time(s) |  Speed(KB/s)
--------------------------------------------------------------
          16 |       16 |  True |   0.000554 |        65.28
          32 |       32 |  True |   0.000820 |        82.22
          64 |       64 |  True |   0.001218 |       106.62
         128 |      128 |  True |   0.002283 |       111.63
         256 |      256 |  True |   0.004242 |       119.01
         512 |      512 |  True |   0.008199 |       122.55
        1024 |     1024 |  True |   0.016056 |       124.87
        2048 |     2048 |  True |   0.031933 |       125.42
        4096 |     4096 |  True |   0.063475 |       126.11
        8192 |     8192 |  True |   0.126813 |       126.21
       16384 |    16384 |  True |   0.253573 |       126.22
       32768 |    32768 |  True |   0.506342 |       126.41


In [8]:
print(f"PORT = {PORT}")
print(f"BAUD = {BAUD}")

TOTAL_TESTS = 100
TEST_SIZE = MAX_BUFFER_SIZE

all_correct = True
success_count = 0
error_count = 0
total_speed = 0.0

for i in range(TOTAL_TESTS):
    msg = os.urandom(TEST_SIZE)
    percent = (i + 1) / TOTAL_TESTS * 100

    try:

        uart.write_line("ECHO_PACKET")
        
        ready = uart.read_line(timeout=2.0)
        if ready != "READY":
            raise RuntimeError(f"Expected READY, got: {ready!r}")
        

        result = uart.echo(msg)

        total_bytes = 2 + TEST_SIZE + 1 + 2 + TEST_SIZE
        speed_kbs = total_bytes / result["elapsed"] / 1024

        if result["ok"]:
            success_count += 1
            total_speed += speed_kbs

            avg_speed = total_speed / success_count

            print(
                f"Progress: {i + 1:3d}/{TOTAL_TESTS} "
                f"({percent:6.2f}%) | "
                f"OK | "
                f"speed={speed_kbs:8.2f} KB/s | "
                f"avg={avg_speed:8.2f} KB/s",
                end="\r",
                flush=True
            )

        else:
            all_correct = False
            error_count += 1
            print()
            print(
                f"ERROR at {i + 1}/{TOTAL_TESTS} "
                f"({percent:.2f}%) | echo mismatch"
            )

    except Exception as e:
        all_correct = False
        error_count += 1
        print()
        print(
            f"ERROR at {i + 1}/{TOTAL_TESTS} "
            f"({percent:.2f}%) | {e}"
        )

    # time.sleep(0.05)

print()

print(f"Success: {success_count}/{TOTAL_TESTS}")
print(f"Errors : {error_count}/{TOTAL_TESTS}")

if all_correct:
    print("NO ERROR")
else:
    print("HAS ERROR")

PORT = /dev/cu.usbmodem1103
BAUD = 1300000
Progress: 100/100 (100.00%) | OK | speed=  126.41 KB/s | avg=  126.34 KB/s
Success: 100/100
Errors : 0/100
NO ERROR


In [9]:
uart.close()